# Model loading #

In [1]:
import torch
import sys
sys.path.insert(0, "./src/glonet")
from modelp2 import Glonet

Error importing huggingface_hub.hf_api: No module named 'filelock'


/Odyssey/private/j25lee/miniforge3/envs/glon/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
if torch.cuda.is_available():
    print("CUDA is available.")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available.")

CUDA is available.
Device name: NVIDIA A100-SXM4-80GB
Device name: NVIDIA A100-SXM4-80GB


In [3]:
model_path = "/Odyssey/public/glonet/TrainedWeights/glonet_part1.pth"

In [4]:
# Check if model file exists
import os
print(f"Model file exists: {os.path.exists(model_path)}")
if os.path.exists(model_path):
    print(f"File size: {os.path.getsize(model_path) / (1024**2):.2f} MB")
else:
    print("Model file not found!")

Model file exists: True
File size: 1081.15 MB


In [5]:
# Method 1: Load TorchScript model (your current approach)
try:
    model = torch.jit.load(model_path, map_location=torch.device('cuda'))
    print("Successfully loaded TorchScript model")
    print(f"Model type: {type(model)}")
except Exception as e:
    print(f"Failed to load as TorchScript: {e}")
    
    # Method 2: Try loading as regular PyTorch state dict
    try:
        checkpoint = torch.load(model_path, map_location=torch.device('cuda'))
        print(f"Successfully loaded checkpoint. Keys: {list(checkpoint.keys()) if isinstance(checkpoint, dict) else 'Not a dictionary'}")
        
        # If it's a state dict, you'll need to create the model first
        if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
            print("This appears to be a checkpoint with state_dict")
            # You would need: model = Glonet(...); model.load_state_dict(checkpoint['state_dict'])
        elif isinstance(checkpoint, dict):
            print("This appears to be a raw state_dict")
            # You would need: model = Glonet(...); model.load_state_dict(checkpoint)
        else:
            model = checkpoint
            print("Loaded as direct model object")
            
    except Exception as e2:
        print(f"Failed to load as regular checkpoint: {e2}")

Failed to load as TorchScript: PytorchStreamReader failed locating file constants.pkl: file not found
Successfully loaded checkpoint. Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'best_loss', 't_loss']
This appears to be a raw state_dict
Successfully loaded checkpoint. Keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'scaler_state_dict', 'best_loss', 't_loss']
This appears to be a raw state_dict


In [6]:
# Correct way to load this model
print("Loading checkpoint...")
checkpoint = torch.load(model_path, map_location=torch.device('cuda'))

# Create the model instance with the correct shape_in parameter
# shape_in = (T, C, H, W) where:
# T = time steps (2), C = channels (5), H = height (672), W = width (1440)
shape_in = (2, 5, 672, 1440)

print(f"Creating Glonet model with shape_in: {shape_in}")
model = Glonet(shape_in=shape_in)

# Load the state dict into the model
print("Loading state dictionary...")
model.load_state_dict(checkpoint['model_state_dict'])

# Set model to evaluation mode 
model.eval()

print("Model loaded successfully!")
print(f"Training epoch: {checkpoint['epoch']}")
print(f"Best loss: {checkpoint['best_loss']}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model is on CUDA: {next(model.parameters()).is_cuda}")

Loading checkpoint...
Creating Glonet model with shape_in: (2, 5, 672, 1440)
720
Creating Glonet model with shape_in: (2, 5, 672, 1440)
720
Loading state dictionary...
Model loaded successfully!
Training epoch: 26
Best loss: 0.016804019961691365
Model parameters: 132,680,155
Model is on CUDA: True
Loading state dictionary...
Model loaded successfully!
Training epoch: 26
Best loss: 0.016804019961691365
Model parameters: 132,680,155
Model is on CUDA: True


In [16]:
# Example of how to use the model for inference
print("Model is ready for inference!")
print(f"Expected input shape: [batch_size, {shape_in[0]}, {shape_in[1]}, {shape_in[2]}, {shape_in[3]}]")
print(f"Input format: [B, T, C, H, W] where:")
print(f"  B = batch size")
print(f"  T = time steps ({shape_in[0]})")
print(f"  C = channels ({shape_in[1]})")  
print(f"  H = height ({shape_in[2]})")
print(f"  W = width ({shape_in[3]})")

# Example with random data (replace with your actual data)
with torch.no_grad():
    # Create dummy input tensor
    dummy_input = torch.randn(1, *shape_in).cuda()  # batch_size=1
    print(f"\nDummy input shape: {dummy_input.shape}")
    
    # Forward pass
    print("Running forward pass...")
    output = model(dummy_input)
    print(f"Output shape: {output.shape}")
    print("Inference completed successfully!")

Model is ready for inference!
Expected input shape: [batch_size, 2, 5, 672, 1440]
Input format: [B, T, C, H, W] where:
  B = batch size
  T = time steps (2)
  C = channels (5)
  H = height (672)
  W = width (1440)

Dummy input shape: torch.Size([1, 2, 5, 672, 1440])
Running forward pass...
Output shape: torch.Size([1, 2, 5, 672, 1440])
Inference completed successfully!
Output shape: torch.Size([1, 2, 5, 672, 1440])
Inference completed successfully!


# Investigate grandient flow #

In [ ]:
# Freeze parameters so only input can receive gradients
for p in model.parameters():
    p.requires_grad = False
model.eval()


In [ ]:
# Create initial condition tensor that requires gradients
import torch.nn as nn

# Initialize x_0 with some initial values (you can modify this)
# Shape: [batch_size, T, C, H, W] = [1, 2, 5, 672, 1440]
x_0 = torch.randn(1, *shape_in, device='cuda', requires_grad=True)

print(f"Initial condition shape: {x_0.shape}")
print(f"Initial condition requires_grad: {x_0.requires_grad}")
print(f"Initial condition is on CUDA: {x_0.is_cuda}")
print(f"Initial condition memory usage: {x_0.numel() * x_0.element_size() / (1024**3):.2f} GB")

In [ ]:
# Define a target or loss function for optimization
# You can modify this based on your specific optimization goal

def compute_loss(model_output, target=None):
    """
    Define your loss function here. Examples:
    1. Minimize specific output values
    2. Match a target pattern
    3. Minimize energy/magnitude
    4. Regularization terms
    """
    
    if target is not None:
        # MSE loss against target
        return nn.MSELoss()(model_output, target)
    else:
        # Example: minimize the mean squared value of the output
        return torch.mean(model_output ** 2)
        
        # Alternative examples:
        # return torch.mean(torch.abs(model_output))  # L1 loss
        # return torch.std(model_output)  # Minimize variance
        # return -torch.mean(model_output)  # Maximize mean output

# Example target (optional - set to None if you don't have a specific target)
target = None  # torch.zeros_like(x_0)  # Uncomment if you want to optimize towards zero

print("Loss function defined!")
print(f"Target: {'None (will minimize output magnitude)' if target is None else target.shape}")

In [4]:
# Test gradient flow - run one forward and backward pass
print("Testing gradient flow...")

# Forward pass
output = model(x_0)
print(f"Forward pass successful! Output shape: {output.shape}")

# Compute loss
loss = compute_loss(output, target)
print(f"Loss computed: {loss.item():.6f}")

# Backward pass
loss.backward()

# Check if gradients are computed
if x_0.grad is not None:
    print(f"✅ Gradients successfully computed!")
    print(f"Gradient shape: {x_0.grad.shape}")
    print(f"Gradient norm: {torch.norm(x_0.grad).item():.6f}")
    print(f"Gradient mean: {torch.mean(x_0.grad).item():.6f}")
    print(f"Gradient std: {torch.std(x_0.grad).item():.6f}")
else:
    print("❌ No gradients computed - check the computational graph!")
    
# Clear gradients for next iteration
x_0.grad.zero_() if x_0.grad is not None else None

Testing gradient flow...


NameError: name 'model' is not defined

# Override Glonet Forward Method Without Detach

In [7]:
import gc

# Create a custom Glonet class that overrides the forward method without detach() calls
class GlonetNoDetach(Glonet):
    def __init__(self, shape_in, hid_S=256, hid_T=128, N_S=2, N_T=8, incep_ker=[3,5,7,11], groups=8):
        super().__init__(shape_in, hid_S, hid_T, N_S, N_T, incep_ker, groups)
    
    def forward(self, input_st_tensors):
        B, T, C, H, W = input_st_tensors.shape
        skip_feature = self.jump(input_st_tensors.to('cuda:0')).contiguous()
        spatial_feature = self.space(input_st_tensors.to('cuda:0')).contiguous()
        # skip_feature=skip_feature.detach()
        # spatial_feature=spatial_feature.detach()
        del input_st_tensors
        gc.collect()  # Force garbage collection
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() 
        spatial_feature = spatial_feature.reshape(-1, C, H, W).contiguous()
        #spatial_embed, spatial_skip_feature=deepspeed.checkpointing.checkpoint(self.latent_projection,spatial_feature)
        spatial_embed, spatial_skip_feature = self.maps(spatial_feature.to('cuda:0'))
        del spatial_feature
        gc.collect()  # Force garbage collection
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() 
        # spatial_embed=spatial_embed.detach()
        # spatial_skip_feature=spatial_skip_feature.detach()
        spatial_embed = spatial_embed.contiguous()
        spatial_skip_feature = spatial_skip_feature.contiguous()
        _, C_, H_, W_ = spatial_embed.shape
        spatial_embed = spatial_embed.view(B, T, C_, H_, W_).contiguous()
        spatialtemporal_embed = self.dynamics(spatial_embed.to('cuda:0')).contiguous()
        # spatialtemporal_embed = spatialtemporal_embed.detach()
        #spatialtemporal_embed=deepspeed.checkpointing.checkpoint(self.TeDev_block,spatial_embed)
        spatialtemporal_embed = spatialtemporal_embed.reshape(B*T, C_, H_, W_).contiguous()
        del spatial_embed
        gc.collect()  # Force garbage collection
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() 
        #predictions=deepspeed.checkpointing.checkpoint(self.dec, spatialtemporal_embed, spatial_skip_feature)
        predictions = self.mapsback(spatialtemporal_embed.to('cuda:0'), spatial_skip_feature.to('cuda:0')).contiguous()
        del spatial_skip_feature, spatialtemporal_embed
        gc.collect()  # Force garbage collection
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() 
        # predictions = predictions.detach()
        predictions = 0.05 * predictions.reshape(B, T, C, H, W).contiguous() + skip_feature.to('cuda:0')
        del skip_feature
        gc.collect()  # Force garbage collection
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() 
        
        return predictions.contiguous()

print("✅ GlonetNoDetach class defined!")
print("This version removes all .detach() calls and memory cleanup to maintain gradient flow")

✅ GlonetNoDetach class defined!
This version removes all .detach() calls and memory cleanup to maintain gradient flow


In [8]:
# Create the gradient-friendly model using the overridden class
print("Creating GlonetNoDetach model...")

# Create new model instance with same architecture
gradient_model = GlonetNoDetach(shape_in=shape_in)

# Load the same weights from the original model
gradient_model.load_state_dict(checkpoint['model_state_dict'])
gradient_model.eval()

# Freeze all model parameters - only input will have gradients
for p in gradient_model.parameters():
    p.requires_grad = False

print("✅ Gradient-friendly Glonet model created!")
print(f"Model parameters frozen: {all(not p.requires_grad for p in gradient_model.parameters())}")
print(f"Model is on CUDA: {next(gradient_model.parameters()).is_cuda}")
print("Ready for initial condition optimization!")

Creating GlonetNoDetach model...
720
✅ Gradient-friendly Glonet model created!
Model parameters frozen: True
Model is on CUDA: True
Ready for initial condition optimization!
✅ Gradient-friendly Glonet model created!
Model parameters frozen: True
Model is on CUDA: True
Ready for initial condition optimization!


In [9]:
# Test gradient flow with the new model
print("Testing gradient flow with GlonetNoDetach...")

# Create a smaller test input to avoid memory issues
test_shape = (1, 2, 5, 672, 1440)  # Smaller spatial dimensions for testing
x_test = torch.randn(test_shape, device='cuda', requires_grad=True)

print(f"Test input shape: {x_test.shape}")
print(f"Test input requires_grad: {x_test.requires_grad}")

# Forward pass
output = gradient_model(x_test)
print(f"Forward pass successful! Output shape: {output.shape}")

# Compute simple loss
loss = torch.mean(output ** 2)
print(f"Loss computed: {loss.item():.6f}")

# Backward pass
print("Computing gradients...")
loss.backward()

# Check if gradients are computed
if x_test.grad is not None:
    print(f"✅ Gradients successfully computed!")
    print(f"Gradient shape: {x_test.grad.shape}")
    print(f"Gradient norm: {torch.norm(x_test.grad).item():.6f}")
    print(f"Gradient mean: {torch.mean(x_test.grad).item():.6f}")
    print(f"Gradient std: {torch.std(x_test.grad).item():.6f}")
    
    # Check for non-zero gradients
    non_zero_grads = (x_test.grad != 0).sum().item()
    total_grads = x_test.grad.numel()
    print(f"Non-zero gradients: {non_zero_grads}/{total_grads} ({100*non_zero_grads/total_grads:.2f}%)")
    
    print("\n🎉 SUCCESS! Gradient flow is now working without detach() calls!")
else:
    print("❌ No gradients computed - there may still be issues in the gradient graph")

# Clean up test variables
del x_test, output, loss
torch.cuda.empty_cache()

Testing gradient flow with GlonetNoDetach...
Test input shape: torch.Size([1, 2, 5, 672, 1440])
Test input requires_grad: True


OutOfMemoryError: CUDA out of memory. Tried to allocate 474.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 4.94 MiB is free. Including non-PyTorch memory, this process has 79.24 GiB memory in use. Of the allocated memory 77.06 GiB is allocated by PyTorch, and 1.68 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Alternative Memory-Efficient Solutions

Here are several strategies to reduce memory usage while maintaining gradient flow:

In [7]:
# Solution 1: Gradient Checkpointing with PyTorch's Built-in Function
import torch.utils.checkpoint as checkpoint_util

class GlonetGradientCheckpointing(Glonet):
    def __init__(self, shape_in, hid_S=256, hid_T=128, N_S=2, N_T=8, incep_ker=[3,5,7,11], groups=8):
        super().__init__(shape_in, hid_S, hid_T, N_S, N_T, incep_ker, groups)
    
    def forward(self, input_st_tensors):
        B, T, C, H, W = input_st_tensors.shape
        
        # Use gradient checkpointing for memory-intensive operations
        def compute_spatial_features(x):
            skip_feature = self.jump(x)
            spatial_feature = self.space(x)
            return skip_feature, spatial_feature
        
        def compute_latent_features(spatial_feature):
            spatial_feature = spatial_feature.reshape(-1, C, H, W)
            return self.maps(spatial_feature)
        
        def compute_temporal_features(spatial_embed):
            return self.dynamics(spatial_embed)
        
        def compute_predictions(spatialtemporal_embed, spatial_skip_feature):
            return self.mapsback(spatialtemporal_embed, spatial_skip_feature)
        
        # Apply gradient checkpointing to reduce memory usage
        skip_feature, spatial_feature = checkpoint_util.checkpoint(
            compute_spatial_features, input_st_tensors, use_reentrant=False
        )
        
        spatial_embed, spatial_skip_feature = checkpoint_util.checkpoint(
            compute_latent_features, spatial_feature, use_reentrant=False
        )
        
        # Reshape for temporal processing
        _, C_, H_, W_ = spatial_embed.shape
        spatial_embed = spatial_embed.view(B, T, C_, H_, W_)
        
        spatialtemporal_embed = checkpoint_util.checkpoint(
            compute_temporal_features, spatial_embed, use_reentrant=False
        )
        
        # Reshape back
        spatialtemporal_embed = spatialtemporal_embed.reshape(B*T, C_, H_, W_)
        
        predictions = checkpoint_util.checkpoint(
            compute_predictions, spatialtemporal_embed, spatial_skip_feature, use_reentrant=False
        )
        
        # Final computation
        predictions = 0.05 * predictions.reshape(B, T, C, H, W) + skip_feature
        
        return predictions

print("✅ GlonetGradientCheckpointing class defined!")
print("This uses PyTorch's gradient checkpointing to trade compute for memory")

✅ GlonetGradientCheckpointing class defined!
This uses PyTorch's gradient checkpointing to trade compute for memory


In [8]:

# Clear PyTorch cache
torch.cuda.empty_cache()


In [9]:
# Create new model instance with same architecture
gradcheck_model = GlonetGradientCheckpointing(shape_in=shape_in)

# Load the same weights from the original model
gradcheck_model.load_state_dict(checkpoint['model_state_dict'])
gradcheck_model.eval()

# Freeze all model parameters - only input will have gradients
for p in gradcheck_model.parameters():
    p.requires_grad = False

# Create a smaller test input to avoid memory issues
test_shape = (1, 2, 5, 672, 1440)  # Smaller spatial dimensions for testing
x_test = torch.randn(test_shape, device='cuda', requires_grad=True)

print(f"Test input shape: {x_test.shape}")
print(f"Test input requires_grad: {x_test.requires_grad}")

# Forward pass
output = gradcheck_model(x_test)
print(f"Forward pass successful! Output shape: {output.shape}")

# Compute simple loss
loss = torch.mean(output ** 2)
print(f"Loss computed: {loss.item():.6f}")

# Backward pass
print("Computing gradients...")
loss.backward()

# Check if gradients are computed
if x_test.grad is not None:
    print(f"✅ Gradients successfully computed!")
    print(f"Gradient shape: {x_test.grad.shape}")
    print(f"Gradient norm: {torch.norm(x_test.grad).item():.6f}")
    print(f"Gradient mean: {torch.mean(x_test.grad).item():.6f}")
    print(f"Gradient std: {torch.std(x_test.grad).item():.6f}")
    
    # Check for non-zero gradients
    non_zero_grads = (x_test.grad != 0).sum().item()
    total_grads = x_test.grad.numel()
    print(f"Non-zero gradients: {non_zero_grads}/{total_grads} ({100*non_zero_grads/total_grads:.2f}%)")
    
    print("\n🎉 SUCCESS! Gradient flow is now working without detach() calls!")
else:
    print("❌ No gradients computed - there may still be issues in the gradient graph")

# Clean up test variables
del x_test, output, loss
torch.cuda.empty_cache()


720
Test input shape: torch.Size([1, 2, 5, 672, 1440])
Test input requires_grad: True
Test input shape: torch.Size([1, 2, 5, 672, 1440])
Test input requires_grad: True
Forward pass successful! Output shape: torch.Size([1, 2, 5, 672, 1440])
Forward pass successful! Output shape: torch.Size([1, 2, 5, 672, 1440])
Loss computed: 0.554936
Computing gradients...
Loss computed: 0.554936
Computing gradients...
✅ Gradients successfully computed!
Gradient shape: torch.Size([1, 2, 5, 672, 1440])
Gradient norm: 0.000796
Gradient mean: 0.000000
Gradient std: 0.000000
Non-zero gradients: 9676800/9676800 (100.00%)

🎉 SUCCESS! Gradient flow is now working without detach() calls!
✅ Gradients successfully computed!
Gradient shape: torch.Size([1, 2, 5, 672, 1440])
Gradient norm: 0.000796
Gradient mean: 0.000000
Gradient std: 0.000000
Non-zero gradients: 9676800/9676800 (100.00%)

🎉 SUCCESS! Gradient flow is now working without detach() calls!
